# 📄 College Project: Resume Analyzer & Career Advisor Agent
### Built with LangChain + LangGraph (Agentic AI)



## Project idea (what you tell your professor 😄)

> *"An agentic AI system that analyzes a student's resume, extracts their skills, compares them against a target job role, computes a career match score, and generates a personalized career report with either project recommendations or a learning roadmap."*

## The multi-agent pipeline

```
Resume → [Skill Extractor] → [Gap Analyzer] → [Match Scorer] → (decision)
                                                                  ├─ score ≥ 60 → [Project Recommender]
                                                                  └─ score < 60 → [Learning Path Agent]
                                                                          ↓
                                                                 [Report Generator] → 📋 Career Report
```

Five agents, one decision point.  **sequential pipelines** + **conditional routing** — the two core patterns of agentic AI.


In [ ]:
# Install libraries (run once)
%pip install -q langgraph langchain

## 1. The State — the shared "case file"

Every agent reads this state and adds its own findings, like doctors adding notes to one patient chart.

In [ ]:
from typing import TypedDict

class CareerState(TypedDict):
    resume_text: str        # input: the student's resume
    target_role: str        # input: the job they want
    skills_found: list      # Agent 1 writes this
    missing_skills: list    # Agent 2 writes this
    match_score: int        # Agent 3 writes this (0-100)
    recommendations: list   # Agent 4a or 4b writes this
    final_report: str       # Agent 5 writes this

print("State defined ✅")

## 2. The knowledge base

A small database of skills per job role. In a real project you could load this from a CSV or an API — for a demo, a dictionary is perfect.

In [ ]:
ROLE_REQUIREMENTS = {
    "data scientist": ["python", "pandas", "numpy", "machine learning", "sql", "statistics"],
    "web developer":  ["html", "css", "javascript", "react", "node", "sql"],
    "ai engineer":    ["python", "machine learning", "deep learning", "langchain", "transformers", "nlp"],
}

# All skills our extractor can recognize (union of everything above + extras)
SKILL_DATABASE = sorted({s for skills in ROLE_REQUIREMENTS.values() for s in skills} |
                        {"java", "c++", "git", "excel", "communication"})

print(f"Known roles: {list(ROLE_REQUIREMENTS)}")
print(f"Recognizable skills: {SKILL_DATABASE}")

## 3. Agent 1: Skill Extractor 🔍

Scans the resume text and picks out every skill it recognizes.

In [ ]:
def skill_extractor(state: CareerState) -> dict:
    """Agent 1: Find known skills mentioned in the resume."""
    resume = state["resume_text"].lower()
    found = [skill for skill in SKILL_DATABASE if skill in resume]

    print(f"🔍 [Skill Extractor] Found {len(found)} skills: {found}")
    return {"skills_found": found}

## 4. Agent 2: Skill Gap Analyzer 📊

Compares the student's skills against what the target role requires.

In [ ]:
def gap_analyzer(state: CareerState) -> dict:
    """Agent 2: Which required skills are missing?"""
    required = ROLE_REQUIREMENTS.get(state["target_role"].lower(), [])
    missing = [s for s in required if s not in state["skills_found"]]

    print(f"📊 [Gap Analyzer] Missing for '{state['target_role']}': {missing if missing else 'none! 🎉'}")
    return {"missing_skills": missing}

## 5. Agent 3: Match Scorer 🎯

Computes a simple percentage: how many required skills the student already has.

In [ ]:
def match_scorer(state: CareerState) -> dict:
    """Agent 3: Career match score = % of required skills present."""
    required = ROLE_REQUIREMENTS.get(state["target_role"].lower(), [])
    if not required:
        score = 0
    else:
        have = len(required) - len(state["missing_skills"])
        score = round(100 * have / len(required))

    print(f"🎯 [Match Scorer] Career match score: {score}/100")
    return {"match_score": score}

## 6. The decision point (conditional routing) 🔀

This is what makes the system *agentic* — it chooses its next step based on the score:

- **Score ≥ 60** → student is nearly ready → recommend **portfolio projects**
- **Score < 60** → big gaps → build a **learning roadmap** first

In [ ]:
def route_by_score(state: CareerState) -> str:
    """Router: decide the next agent based on the match score."""
    return "project_recommender" if state["match_score"] >= 60 else "learning_path_agent"


def project_recommender(state: CareerState) -> dict:
    """Agent 4a: Suggest portfolio projects using skills they HAVE."""
    recs = [f"Build a project combining {s} with your other skills" for s in state["skills_found"][:3]]
    recs.append(f"Publish your work on GitHub and write a short blog about it")

    print("💡 [Project Recommender] Student is nearly job-ready → suggesting projects")
    return {"recommendations": recs}


def learning_path_agent(state: CareerState) -> dict:
    """Agent 4b: Build a learning roadmap for the MISSING skills."""
    recs = [f"Week {i+1}-{i+2}: Learn '{skill}' (free course + one mini exercise)"
            for i, skill in enumerate(state["missing_skills"])]

    print("🗺️ [Learning Path Agent] Gaps found → building a roadmap")
    return {"recommendations": recs}

## 7. Agent 5: Report Generator 📋

Assembles everything into a readable career report. Both branches merge back into this node.

In [ ]:
def report_generator(state: CareerState) -> dict:
    """Agent 5: Assemble the final career report."""
    lines = [
        "=" * 55,
        f"        AI CAREER REPORT — Target: {state['target_role'].title()}",
        "=" * 55,
        f"✅ Skills found ({len(state['skills_found'])}): {', '.join(state['skills_found'])}",
        f"❌ Missing skills: {', '.join(state['missing_skills']) if state['missing_skills'] else 'None'}",
        f"🎯 Match score: {state['match_score']}/100",
        "",
        "📌 RECOMMENDATIONS:",
    ]
    lines += [f"   {i+1}. {r}" for i, r in enumerate(state["recommendations"])]

    print("📋 [Report Generator] Report ready!")
    return {"final_report": "\n".join(lines)}

## 8. Wire the agents into a graph

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(CareerState)

builder.add_node("skill_extractor", skill_extractor)
builder.add_node("gap_analyzer", gap_analyzer)
builder.add_node("match_scorer", match_scorer)
builder.add_node("project_recommender", project_recommender)
builder.add_node("learning_path_agent", learning_path_agent)
builder.add_node("report_generator", report_generator)

# Sequential part
builder.add_edge(START, "skill_extractor")
builder.add_edge("skill_extractor", "gap_analyzer")
builder.add_edge("gap_analyzer", "match_scorer")

# Decision point
builder.add_conditional_edges("match_scorer", route_by_score, {
    "project_recommender": "project_recommender",
    "learning_path_agent": "learning_path_agent",
})

# Both branches merge into the report
builder.add_edge("project_recommender", "report_generator")
builder.add_edge("learning_path_agent", "report_generator")
builder.add_edge("report_generator", END)

career_agent = builder.compile()
print("Career advisor agent compiled ✅")

In [ ]:
# Visualize the agent workflow
try:
    from IPython.display import Image, display
    display(Image(career_agent.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: paste this into https://mermaid.live to view the diagram
    print(career_agent.get_graph().draw_mermaid())

## 9. Demo 1: A strong candidate (takes the *project* branch)

In [ ]:
strong_resume = """
B.Tech CSE student. Skilled in Python, Pandas, NumPy and SQL.
Completed a Machine Learning course project on house price prediction.
Strong foundation in Statistics. Familiar with Git and Excel.
"""

result = career_agent.invoke({
    "resume_text": strong_resume, "target_role": "data scientist",
    "skills_found": [], "missing_skills": [], "match_score": 0,
    "recommendations": [], "final_report": "",
})
print()
print(result["final_report"])

## 10. Demo 2: A beginner (takes the *learning roadmap* branch)

Same graph, same code — the agent **decides differently** because the input is different.

In [ ]:
beginner_resume = """
First year student. Knows Java basics and C++ from class.
Good communication skills. Interested in becoming an AI engineer.
"""

result = career_agent.invoke({
    "resume_text": beginner_resume, "target_role": "ai engineer",
    "skills_found": [], "missing_skills": [], "match_score": 0,
    "recommendations": [], "final_report": "",
})
print()
print(result["final_report"])

## 11. Try YOUR resume ✏️

Paste your own resume text and pick a role: `data scientist`, `web developer`, or `ai engineer`.

In [ ]:
my_resume = """
Paste your resume text here...
"""                                # ← edit me!
my_target = "data scientist"       # ← edit me!

result = career_agent.invoke({
    "resume_text": my_resume, "target_role": my_target,
    "skills_found": [], "missing_skills": [], "match_score": 0,
    "recommendations": [], "final_report": "",
})
print()
print(result["final_report"])

## 12. Upgrading to a real LLM (for the final-year version)

Swap any agent's insides for an LLM call — the graph doesn't change:

```python
# pip install langchain-openai   (needs an API key)
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

def skill_extractor(state: CareerState) -> dict:
    reply = llm.invoke(f"List the technical skills in this resume as comma-separated values: {state['resume_text']}")
    return {"skills_found": [s.strip().lower() for s in reply.content.split(",")]}
```

